# Density of States Effective Mass with VASP

The DOS effective mass is necessary for the final PV Figure of Merit. There are two approaches to creating a suitable DFT input: 

1) Perform a dense single-point DOS calculation, with a large `NEDOS`. This is more straightfoward but more computationally expensive.

2) A standard single-point DOS with additional points around the location of the band edge(s).

#### Pre-requisites:
* Structural Relaxation
* Band Structure - to choose the band-edge central point(s)

1) Import the necessary libraries

In [1]:
import numpy as np

import solphin.band_structure as band_structure
import solphin.dos as dos
import solphin.vasp_inputs as vasp_inputs

## Method 1

1) Define our computational regime. This is the same as used to relax the structure, except with a denser k-point mesh.

In [ ]:
functional = 'HSE06'  # The DFT functional
encut = 450           # Plane-wave basis set energy cutoff
kspacing = 0.1        # k-point sampling density, this is just an example value

2) Load the optimised atomic structure from a previous relaxation. Here we read from the bundled test data. 

In [10]:
structure = vasp_inputs.read_structure_pmg("../tests/data/Cu2GeS3/Relax/CONTCAR")

3. Ensure the structure has the same canonical orientation used for the band structure.

In [11]:
canonical_structure, _ = band_structure.generate_band_structure_path(structure=structure, definition="bradcrack")

Generated high-symmetry path of 239 k-points


4) Write the calculation inputs.

In [12]:
vasp_inputs.write_vasp_calculation(
    structure=structure,
    recipe=functional,
    out_dir="workdir/eff_mass",
    patches=["dos"],
    user_incar_settings={"KSPACING": kspacing, "ENCUT": encut, "ISPIN": 1, "ISMEAR": -5, "SIGMA": 0.02, "LORBIT": 14, "NEDOS":6000})

## Method 2

1) Define our computational regime. This should be the same regime used to relax the structure.

In [2]:
functional = 'HSE06'  # The DFT functional
encut = 450           # Plane-wave basis set energy cutoff

2) Load the optimised atomic structure from a previous relaxation. Here we read from the bundled test data. 

In [3]:
structure = vasp_inputs.read_structure_pmg("../tests/data/Cu2GeS3/Relax/CONTCAR")

3. Ensure the structure has the same canonical orientation used for the band structure.

In [4]:
canonical_structure, _ = band_structure.generate_band_structure_path(structure=structure, definition="bradcrack")

Generated high-symmetry path of 239 k-points


4) Load the standard set of irreducible SCF k-points from a previous calculation, the same as when performing a band structure calculation. Here we read from the test data's relaxation.

In [6]:
from pymatgen.io.vasp import Kpoints

irred_kpts = Kpoints.from_file("../tests/data/Cu2GeS3/OPT_hybrid/IBZKPT")

5) Write the VASP input files. Here we write to `workdir/optics` which is untracked, and can be used to try out these tutorials. This requires having set up your VASP `POTCAR`s with `pymatgen`.

* We need to find the k-point coordinates of our VBM/CBM. In this example the gamma point (0,0,0) is the location of our band-edges, as the system is direct gap.
* In the case of an indirect gap system, supply `k0_frac` as a list of points.
* The `mesh` flag controls the number of additional points created around each central point as (nx, ny, nz).
* The `delta` flag controls how far from the central point the additional mesh ranges, in fractional/relative coordinate space. 
* Together, `mesh` and `delta` control the range and density of the sampling.

In [7]:
dos.write_eff_mass(
    k0_frac = np.array([0,0,0]),
    mesh = (5,5,5),
    delta=0.05,
    structure = canonical_structure,
    functional = functional,
    encut = encut,
    irred_kpts=irred_kpts,
    folder = "workdir/eff_mass"
)

6) Run the VASP calculation. The specifics will depend on your particular machine, but will require invoking the `vasp_std` command in a suitable environment.

## Post-Processing

1) Fit the effective masses from the calculation results. We point again to the reference data. 

* This outputs the quality of the fitting as an R² value, so a value that is far from 1 indicates poor/inappropriate sampling of the band edges.

In [8]:
result = dos.compute_dos(filepath=f"../tests/data/Cu2GeS3/DOS_HDFT/vasprun.xml", code="vasp")
print(result)

  Computing electron FOM DOS effective mass...
  Computing hole FOM DOS effective mass...

  DOS Result Summary
  Primary carrier     : Electrons
  Band edge (CBM)     : 1.402 eV
  Cell volume         : 2.234e-28 m³

  ── FOM DOS Effective Masses ───────────────────────
  Electrons (fitted at CBM: 1.402 eV)
  DOS effective mass    : 0.103528 mₑ  (9.431e-32 kg)  ← primary carrier
  Fit quality           : 0.640706 R²
  Points fitted         : 11
  Energy window         : 0.1500 eV


  Holes (fitted at VBM: 0.000 eV)
  DOS effective mass    : 1.080489 mₕ  (9.843e-31 kg)
  Fit quality           : 0.627290 R²
  Points fitted         : 11
  Energy window         : 0.1500 eV


  Γₚᵥ DOS mass √(mₑm_h) : 0.334456 m₀

